# Neural prices and spot risk

The value-gradient solver fits a shared $u_\theta(t,s)$ and differentiates it in physical spot units. The terminal-loss solver learns an initial scalar and Brownian controls. Its initial spot delta solves $\Sigma(s_0)^\top\Delta=\widetilde Z_0$. The checkpoints minimise their own scheduled validation losses.

## Price and delta targets

An accuracy pass requires both an absolute price error of at most 0.01 and an absolute error of at most 0.01 for every $\Delta_i/w_i$, accounting for reference uncertainty. The default tables use the recorded workspace observations. A local suite summary, when present, supplies the table values.


In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
evidence = json.loads((root / "experiments/observations.json").read_text())
summary_path = root / "scratch/runs/final_suite/summary.json"
summary = json.loads(summary_path.read_text()) if summary_path.exists() else None

def show_table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

observed = evidence["fits"]
headers = ["Solver", "Completed fits", "Price passes", "All-delta passes",
           "Joint passes", "Coordinates outside bounds"]
rows = []
for solver, counts in observed["by_solver"].items():
    if summary is None:
        rows.append((solver.replace("_", " "), counts["fits"], counts["price_passes"],
                     counts["all_delta_passes"], counts["joint_passes"],
                     counts["initial_delta_bound_violations"]))
        continue
    fits = [f for f in summary["fits"]
            if f["status"] == "completed" and f["job"]["solver"] == solver]
    rows.append((
        solver.replace("_", " "), len(fits),
        sum(f["price_classification"] == "within_target" for f in fits),
        sum(f["delta_classifications"] == {"within_target": len(f["delta"])} for f in fits),
        sum(f["accuracy"] == "within_target" for f in fits),
        sum(f["initial_bound_violations"] for f in fits),
    ))
show_table(headers, rows)


| Solver | Completed fits | Price passes | All-delta passes | Joint passes | Coordinates outside bounds |
| --- | --- | --- | --- | --- | --- |
| value gradient | 39 | 3 | 0 | 0 | 5 |
| terminal loss | 39 | 14 | 3 | 1 | 144 |


The workspace run completed all 78 fits. One vanilla terminal-loss fit met both targets. Terminal loss passed the price target more often and had more recovered deltas outside the economic bounds. These bounds check nonnegativity and the case-specific delta ceiling.

The eight numerical references all qualified on the first allocation. Their largest price and scaled-delta half-widths were 0.00056 and 0.00057 respectively, within the 0.002 budget.

## Dimension and the time grid

The 100-asset case reports signed price error and the largest absolute coordinate error within each seed. The ten-asset grid comparison keeps training settings and the financial reference fixed.


In [ ]:
hundred = observed["arithmetic_100"]["rows"]
grid = observed["arithmetic_10"]
grid_errors = grid["value_gradient_max_abs_price_error"]
if summary is not None:
    fits = [f for f in summary["fits"] if f["status"] == "completed" and f["scaled_delta_errors"]]
    hundred = [
        (f["job"]["solver"], f["job"]["seed"], f["price_error"],
         max(map(abs, f["scaled_delta_errors"])))
        for f in fits if f["job"]["case"] == "arithmetic-100"
    ]
    grid_errors = [
        max((abs(f["price_error"]) for f in fits
             if (f["job"]["case"], f["job"]["n_steps"], f["job"]["solver"])
             == ("arithmetic-10", steps, "value_gradient")), default=None)
        for steps in grid["grids"]
    ]
show_table(["100-asset solver", "Seed", "Signed price error", "Worst scaled delta error"],
           [(solver.replace("_", " "), seed, f"{price:+.5f}", f"{delta:.5f}")
            for solver, seed, price, delta in hundred])
show_table(["Ten-asset intervals", "Worst value-gradient price error"],
           [(steps, f"{error:.4f}" if error is not None else "—")
            for steps, error in zip(grid["grids"], grid_errors)])


| 100-asset solver | Seed | Signed price error | Worst scaled delta error |
| --- | --- | --- | --- |
| value gradient | 41001 | +0.05559 | 0.38578 |
| value gradient | 41002 | -0.04096 | 0.24542 |
| value gradient | 41003 | -0.07050 | 0.51810 |
| terminal loss | 41001 | -0.03730 | 2.13424 |
| terminal loss | 41002 | -0.06895 | 1.31787 |
| terminal loss | 41003 | -0.07105 | 1.61011 |

| Ten-asset intervals | Worst value-gradient price error |
| --- | --- |
| 10 | 0.1033 |
| 16 | 0.1220 |
| 20 | 0.1820 |
| 40 | 0.1047 |


All six 100-asset price errors exceeded 0.01. The ladder changes diversification along with dimension because the arithmetic baskets use equal weights.

At ten assets, six of the 24 fits met the price target and all 24 missed the joint target. The worst value-gradient price error varied non-monotonically across grids. Exact GBM transitions isolate the grid change to the control integral and learning problem. The terminal formulation also adds a network at each extra time node.

## Vanilla curvature and held-out paths

The value network defines a spot-to-price surface. Its initial gamma can be compared with Black–Scholes. The RMSEs use separate pre-expiry paths.


In [ ]:
vanilla = observed["vanilla_value_gradient"]
rows = vanilla["rows"]
if summary is not None:
    rows = [
        (f["job"]["seed"], f["gamma"]["estimate"], f["path_rmse"]["price_rmse_before_expiry"],
         f["path_rmse"]["delta_rmse_before_expiry"])
        for f in summary["fits"]
        if f["job"]["case"] == "vanilla" and f["status"] == "completed" and f.get("gamma")
    ]
show_table(["Seed", "Gamma", "Exact gamma", "Path price RMSE", "Path delta RMSE"],
           [(seed, f"{gamma:.6f}", f"{vanilla['exact_gamma']:.6f}", f"{price:.4f}", f"{delta:.4f}")
            for seed, gamma, price, delta in rows])


| Seed | Gamma | Exact gamma | Path price RMSE | Path delta RMSE |
| --- | --- | --- | --- | --- |
| 41001 | 0.004682 | 0.019333 | 0.9233 | 0.1028 |
| 41002 | 0.002828 | 0.019333 | 0.9594 | 0.1100 |
| 41003 | 0.002002 | 0.019333 | 0.9499 | 0.1108 |


The fitted gammas range from 0.0020 to 0.0047 against the exact 0.0193. That curvature error is an initial-state diagnostic. The path RMSEs measure prices and deltas across the states reached before expiry.
